In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import yaml
import lightning_scripts.lightning_classifier_matched_speech_in_noise as lightning 
import importlib
import torch
from lightning_scripts import jsinV3DataLoader_precombined_batched 
import pandas as pd 

In [2]:
torch.set_float32_matmul_precision = 'medium'

In [3]:
x = torch.ones(3,1,5)
y = torch.zeros(3,1,5)
y[1,:] = 1 

torch.eq(x,y).all(1)

tensor([[False, False, False, False, False],
        [ True,  True,  True,  True,  True],
        [False, False, False, False, False]])

In [3]:
###
config_path = "model_configs/word_speaker_audioset_resnet18_MatchedDataset_AdamW_shuffle_train.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 2
config['hparas']['batch_size'] = 16
config['hparas']['lr'] = .001
# config['data']['overfit'] = True

In [6]:
importlib.reload(lightning)
LitWordAudioSetModel = lightning.LitWordAudioSetModel
from lightning.pytorch.loggers import WandbLogger

model = LitWordAudioSetModel(config)


In [6]:
train_loader = model.train_dataloader()

In [7]:
# np.random.seed(0)
# prev_batch = None
# for ix, batch in enumerate(train_loader):
#     if ix == 0:
#         prev_batch = batch[1]['signal/word_int']
#     if ix > 0:
#         cur_batch = batch[1]['signal/word_int']
#         print(len(set(cur_batch.tolist()).intersection(set(prev_batch.tolist()))))
#         prev_batch = cur_batch

#     if ix == 10:
#         break

### Try Lightning Module

In [7]:

# wandb_logger = WandbLogger(project="dev_matched_supervised")
# wandb_logger.watch(model.model, log="all", log_freq=5)


trainer = L.Trainer(
    # limit_train_batches=5,
    # limit_val_batches=0,
    max_epochs=config['hparas']['epochs'],
    # strategy='ddp_notebook',
    # gradient_clip_val=1,
    # overfit_batches=.001,
    # logger=wandb_logger,
    devices=1)

trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          | 12.1 M | train
2 | multi_task_loss | jsinV3_multi_task_loss     | 0      | train
3 | train_accuracy  | ModuleDict                 | 0      | train
4 | val_accuracy    | ModuleDict                 | 0      | train
----------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

## Debug Matched dataset 

In [9]:
config['data']['target_keys']

['signal/word_int', 'signal/speaker_int', 'noise/labels_int']

In [10]:
config['data']['val_noise_h5_path']

'/mnt/home/jfeather/ceph/data/training_datasets_audio/audioset_dataframes/sr20000/cullLabels_silence/sr20000_balanced_train_segments_raw_exclude_speech_and_only_music_maxZerosPercent10.pdh5'

In [11]:
importlib.reload(jsinV3DataLoader_precombined_batched)

### Write collate function 
def collate_fn(batch):
    batch = batch[0]
    if len(batch) == 2:
        all_audio = []
        for audio in batch[0]:
            all_audio.append(audio.unsqueeze(1))
        labels = []
        for label_set in batch[1]:
            view_labels = {}
            if isinstance(label_set, dict):
                for key, l in label_set.items():
                    view_labels[key] = l.squeeze()
                labels.append(view_labels)
            else:
                labels.append(label_set.squeeze())
        return all_audio, labels 
    elif len(batch) == 4: # no labels
        return [audio.unsqueeze(1) for audio in batch]
            

batch_size = 4

MatchedSpeechInNoiseDatasetBatched = jsinV3DataLoader_precombined_batched.MatchedSpeechInNoiseDatasetBatched
dataset = MatchedSpeechInNoiseDatasetBatched(config['data']['speech_h5_path'],
                                             config['data']['noise_h5_path'],
                                             target_keys=config['data']['target_keys'],
                                             batch_size=batch_size)
### Build audioset class map from ix to human readable 

audioset_label_dict = {}
for [labels, label_ixs] in dataset.noise_metadata[['labels_decoded_slugged', 'labels_int']].values:
    unique_labels = labels.split(',')
    for label, class_ix in zip(*[unique_labels, label_ixs]):
        if label in audioset_label_dict.values():
            continue
        audioset_label_dict[class_ix.item()] = label
word_label_dict, _ = dataset.class_map()
# audio, labels = collate_fn(dataset[0])
dataloader = torch.utils.data.DataLoader(dataset, collate_fn=collate_fn)

## Make sure noise files actually read and indexed correctly

In [12]:
ix = 345

print(dataset.noise_metadata[['labels_decoded_slugged', 'labels_int']].iloc[ix])
audio = dataset.noise_files['ndarray_data']['signal'][ix]
Audio(audio, rate=20000)

labels_decoded_slugged    accelerating-revving-vroom,vehicle
labels_int                                        [291, 379]
Name: 345, dtype: object


In [13]:
config['data']['noise_h5_path']

'/mnt/home/jfeather/ceph/data/training_datasets_audio/audioset_dataframes/sr20000/cullLabels_silence/sr20000_unbalanced_train_segments_raw_exclude_speech_and_only_music_maxZerosPercent10.pdh5'

In [14]:
#### Make sure the rows in the h5 array matches the manifest 

manifest_labels = dataset.noise_metadata['labels_int'][:100].values
manifest_one_hot = np.zeros((len(manifest_labels), 517))
for ix, label in enumerate(manifest_labels):
    manifest_one_hot[ix, label] = 1 

assert all(manifest_one_hot.sum(1) == np.array([len(labels) for labels in manifest_labels])), "Label missmatch "

h5_one_hot = np.vstack(dataset.noise_files['ndarray_data']['labels_binary_via_int'][:100])
assert (h5_one_hot == manifest_one_hot).all(), "Missmatch in manifest and h5 array"

In [15]:
#### Make sure background pairing is correct 
np.random.seed(0)
SR = 20_000

batch_ix = 100


## Check word balance labeling is correct across batches 
[combo11, combo12, combo21, combo22], [labels11, labels12, labels21, labels22] = collate_fn([dataset[batch_ix]])


get_word = lambda ix: word_label_dict[ix.item()]

def get_noise_str(binary_labels):
    return ", ".join([audioset_label_dict[ix.item()] for ix in binary_labels.argwhere()])

#### Logic, rows of comboX1 and comboX2 should have same speech/talker labels (signal/word_int or signal/speaker_int)
#### rows of combo1X combo2X should have same augmentation, and same noise/label_int 


pair_ix = 2

### unpack signals & labels 

word11 = get_word(labels11['signal/word_int'][pair_ix])
word12 = get_word(labels12['signal/word_int'][pair_ix])
word21 = get_word(labels21['signal/word_int'][pair_ix])
word22 = get_word(labels22['signal/word_int'][pair_ix])

talker11 = labels11['signal/speaker_int'][pair_ix]
talker12 = labels12['signal/speaker_int'][pair_ix]
talker21 = labels21['signal/speaker_int'][pair_ix]
talker22 = labels22['signal/speaker_int'][pair_ix]

noise11 = get_noise_str(labels11['noise/labels_int'][pair_ix])
noise12 = get_noise_str(labels12['noise/labels_int'][pair_ix])
noise21 = get_noise_str(labels21['noise/labels_int'][pair_ix])
noise22 = get_noise_str(labels22['noise/labels_int'][pair_ix])

audio11 = combo11[pair_ix]
audio12 = combo12[pair_ix]
audio21 = combo21[pair_ix]
audio22 = combo22[pair_ix]

print("Audio 11 ")
print(f"\t word: {word11}")
print(f"\t noises: {noise11}")
display(Audio(audio11, rate=SR))
print("Audio 12 ")
print(f"\t word: {word12}")
print(f"\t noises: {noise12}")
display(Audio(audio12, rate=SR))

print("Audio 21 ")
print(f"\t word: {word21}")
print(f"\t noises: {noise21}")
display(Audio(audio21, rate=SR))
print("Audio 22 ")

print(f"\t word: {word22}")
print(f"\t noises: {noise22}")
display(Audio(audio22, rate=SR))

Audio 11 
	 word: twenty
	 noises: rock-music


Audio 12 
	 word: twenty
	 noises: video-game-music


Audio 21 
	 word: century
	 noises: rock-music


Audio 22 
	 word: century
	 noises: video-game-music


### Make sure supervised collate function works 

In [20]:
#### Dev supervised collate function 

def collate_fn(batch):
    audio, targets = batch[0]
    audio = torch.vstack(audio).unsqueeze(1)
    print(audio.shape)
    labels = {}
    for label_key in targets[0].keys():
        labels[label_key] = torch.concat([label_set[label_key] for label_set in targets])
    return audio, labels

In [22]:
audio, labels = collate_fn([dataset[100]])

for ix in range(audio.shape[0]):
    word = get_word(labels['signal/word_int'][ix])
    talker = labels['signal/speaker_int'][ix]
    noise = get_noise_str(labels['noise/labels_int'][ix])
    
    audio_eg = audio[ix]
    print(f"Audio {ix} ")
    print(f"\t word: {word}")
    print(f"\t noises: {noise}")
    display(Audio(audio_eg, rate=SR))


torch.Size([16, 1, 40000])
Audio 0 
	 word: which
	 noises: electronic-music, techno


Audio 1 
	 word: several
	 noises: music-of-asia, music-of-bollywood


Audio 2 
	 word: twenty
	 noises: horse, animal


Audio 3 
	 word: alone
	 noises: vehicle, car


Audio 4 
	 word: which
	 noises: accelerating-revving-vroom


...

Audio 12 
	 word: different
	 noises: accelerating-revving-vroom


Audio 13 
	 word: political
	 noises: singing, inside-large-room-or-hall


Audio 14 
	 word: century
	 noises: afrobeat


Audio 15 
	 word: equipment
	 noises: ambient-music


wandb: 🚀 View run brisk-galaxy-27 at: https://wandb.ai/iangriffith-harvard-university/dev_matched_supervised/runs/xx6q8w9l
